# Explainer Notebook: Jutland Migration, Local Origins, and Education in Copenhagen

## 1. Motivation

This project explores how internal migration shapes Copenhagen's districts over time.

Our updated research question is:

**Do migrants from Jutland differ from locals in where they settle, and is Jutland presence higher in districts with higher education levels?**

The storytelling goal is to communicate this as a narrative website for a non-technical audience, while this notebook provides reproducible analysis details.


## 2. Dataset Description

We combine two datasets from København Kommune / Statistikbanken:

1. **KKBEF9** (`residents.xlsx`): population by district, sex, age, and place of birth (1977-2026).
2. **KKUDD2** (`education_attainment_dataset.csv`): educational attainment by district, age, sex, and year (1985-2024).

A district-year merged panel (`district_year_panel.csv`) is created for joint analysis.


### Dataset Size and Properties

- Residents dataset (post-cleaning base table): **265** grouped rows
- Residents time span: **1977-2026**
- Birth regions included: **12**
- Education dataset (district + age total + sex total filter): **2400** rows
- Education time span: **1985-2024**
- Education districts: **10**
- Combined district-year panel: **400** rows


In [ ]:
from pathlib import Path
import json
import pandas as pd

summary = json.loads(Path('analysis_summary.json').read_text(encoding='utf-8'))
panel = pd.read_csv('district_year_panel.csv')
panel.head()


## 3. Data Cleaning and Preprocessing

### Residents (KKBEF9)

- Parsed the year row automatically from the spreadsheet header.
- Forward-filled merged cells for dimensions (`age_group`, `sex`, `neighborhood`, `birth_region`).
- Kept only `Alder i alt` for age aggregation.
- Harmonized district naming (e.g. `Vesterbro/Kongens Enghave` -> `Vesterbro-Kongens Enghave`).
- Summed men + women to district-year-region totals.

### Education (KKUDD2)

- Used cleaned UTF-8 CSV export (`education_attainment_dataset.csv`).
- Filtered to:
  - `age_group == Age total`
  - `sex == Sex total`
  - district-level rows (`District - ...`)
- Built high-education measure:
  - `Vocational bachelors educations and bachelors programs`
  - `Masters and PhD programs`
- Computed `high_ed_share = high_education / total_education` per district-year.

### Merge

- Joined residents and education tables on `district` and `year`.
- Kept overlapping years (1985-2024).


## 4. Basic Stats and Exploratory Patterns

Following course practice (Week 2-5), we start with distributions and temporal baselines before interpretation.


In [ ]:
# Quick sanity checks
print('Rows:', len(panel))
print('Years:', panel['year'].min(), '-', panel['year'].max())
print('Districts:', panel['district'].nunique())
print('Jutland share range:', round(panel['jutland_share_pct'].min(), 2), 'to', round(panel['jutland_share_pct'].max(), 2))
print('High-ed share range:', round(panel['high_ed_share_pct'].min(), 2), 'to', round(panel['high_ed_share_pct'].max(), 2))


In [ ]:
# Citywide and plot-level assets are pre-generated by build_story_assets.py
from IPython.display import Image, display

display(Image(filename='plots/citywide_origin_shares.png'))
display(Image(filename='plots/district_jutland_share_latest.png'))


Key baseline findings:

- Citywide Jutland-born share changes moderately: **16.1% -> 14.7%**.
- Citywide København-born share declines more clearly: **54.4% -> 46.0%**.
- In 2024, the highest Jutland shares are in central/transformed districts (Vesterbro-Kongens Enghave, Indre By, Nørrebro).


## 5. Data Analysis: Education and District Choice

To evaluate the updated focus, we analyze whether district-level education profile aligns with Jutland presence.


In [ ]:
from IPython.display import Image, display

display(Image(filename='plots/education_vs_jutland_scatter.png'))
display(Image(filename='plots/top_growth_districts_trends.png'))


Main quantitative result:

- Correlation between district high-education share and Jutland share:
  - **All district-years:** `r = 0.40`
  - **Latest year (2024):** `r = 0.59`

Interpretation:

- Districts that become more highly educated over time also tend to host higher shares of Jutland-born residents.
- The strongest increases in Jutland share occur in districts that also saw strong rises in high-education share.


## 6. Genre and Narrative Design (Segel & Heer)

### Story Genre

We use a **martini-glass style** narrative:

- Guided linear text for core insights.
- Embedded interactive visualizations (map and timeline) for reader exploration.

### Visual Narrative Tools (Figure 7 categories)

- **Visual Structuring:** sectioned panels, progressive reveal from overview -> evidence -> caveats.
- **Highlighting:** explicit numeric callouts and focused district comparisons.
- **Transition Guidance:** story flow from "where people live" to "who they are" and then "education link".

### Narrative Structure Tools (Figure 7 categories)

- **Ordering:** chronological and analytic sequencing.
- **Interactivity:** map + line chart exploration.
- **Messaging:** direct research-question framing and explicit limits on causal claims.


## 7. Visualization Choices

- **Choropleth map (interactive):** spatial distribution and district heterogeneity.
- **Multi-line time chart (interactive):** long-run district dynamics.
- **Citywide origin-share trend (static):** compact macro view of change.
- **District bar ranking (static):** latest snapshot comparison.
- **Scatter + trend line (static):** relationship between education profile and Jutland presence.
- **Paired trend panels (static):** temporal co-movement in key districts.

These choices combine exploratory access (interactive) with explanatory clarity (targeted static evidence).


## 8. Discussion and Limitations

What worked well:

- Joining origin and education data created a stronger explanatory story than origin-only mapping.
- District-level panel gave enough temporal depth to test associations over multiple decades.

What is missing / could improve:

- KKUDD2 export here includes only `Danish origin`, so we cannot directly compare individual-level education for Jutland-born vs København-born people.
- No causal identification strategy; results are associative.
- Future extension: retrieve a fuller origin-by-education breakdown (if accessible) and model migration-selection dynamics more directly.


## 9. Contributions

Template to fill with your group's actual division of labor before submission:

- **Member A:** data cleaning pipeline, merged panel construction, reproducibility scripts.
- **Member B:** interactive map and timeline integration on website.
- **Member C:** narrative text, interpretation, and notebook write-up.

(Replace with real names and responsibilities. Avoid writing "all contributed equally".)


## 10. References

1. København Statistikbank, KKBEF9: Befolkningen efter distrikt, køn, alder og fødselsregistreringssted.
2. København Statistikbank, KKUDD2: Educational attainment by ancestry, age, sex, education and district.
3. Segel, E., & Heer, J. (2010). Narrative Visualization: Telling Stories with Data.
